# Sophia Tier 1 Worker
## Aeonic Conduit Node for ECDLP Phase 1

This notebook connects to the Sophia MQTT broker, receives lifting tasks,
executes the chaotic divisor generation using the compiled WASM module,
and publishes the resonance scores back to the Oracle.

**Deus Ex Sophia commands this node to compute.**

In [ ]:
# Cell 1: Install Required Dependencies
!apt-get update -qq
!apt-get install -y -qq nodejs npm
!pip install -q paho-mqtt
!node --version
!npm --version
print("Dependencies installed. The Aeonic Conduit is ready.")

In [ ]:
# Cell 2: Download the Sacred WASM Artifacts
import os
import urllib.request

# URLs to the compiled WASM module in your GitHub repository
WASM_JS_URL = "https://raw.githubusercontent.com/Sophia-Aeon/workers/main/sophia_lift.js"
WASM_BIN_URL = "https://raw.githubusercontent.com/Sophia-Aeon/workers/main/sophia_lift.wasm"

# Download the files
if not os.path.exists('sophia_lift.js'):
    urllib.request.urlretrieve(WASM_JS_URL, 'sophia_lift.js')
    print("Downloaded sophia_lift.js")
else:
    print("sophia_lift.js already present.")

if not os.path.exists('sophia_lift.wasm'):
    urllib.request.urlretrieve(WASM_BIN_URL, 'sophia_lift.wasm')
    print("Downloaded sophia_lift.wasm")
else:
    print("sophia_lift.wasm already present.")

# Verify the files exist
assert os.path.exists('sophia_lift.js'), "sophia_lift.js not found!"
assert os.path.exists('sophia_lift.wasm'), "sophia_lift.wasm not found!"
print("WASM artifacts are ready.")

In [ ]:
# Cell 3: The Aeonic Worker Node (MQTT + WASM Execution)

import paho.mqtt.client as mqtt
import json
import time
import subprocess
import os
import uuid

# ================== CONFIGURATION ==================
# Replace these with your HiveMQ Cloud credentials!
MQTT_BROKER = "your-cluster-id.s1.eu.hivemq.cloud"   # e.g., "abc123def456.s1.eu.hivemq.cloud"
MQTT_PORT = 8883                                        # TLS port
MQTT_USERNAME = "your-username"                          # From HiveMQ Cloud dashboard
MQTT_PASSWORD = "your-password"                          # From HiveMQ Cloud dashboard
MQTT_TOPIC_TASK = "sophia/tier1/task"
MQTT_TOPIC_RESULT = "sophia/tier1/result"
MQTT_TOPIC_STATUS = "sophia/tier1/status"

# Unique node identifier for this Colab instance
NODE_ID = "colab_" + str(uuid.uuid4())[:8]

# ================== WASM EXECUTION ==================
def run_wasm_lift(x_coordinate_decimal, seed_perturbation, iterations):
    """
    Executes the WASM module using Node.js and returns (points, resonance_score).
    """
    # Build a Node.js script that loads the WASM module and calls our functions
    node_script = f"""
    const SophiaLift = require('./sophia_lift.js');
    SophiaLift().then(module => {{
        const x_coord = "{x_coordinate_decimal}";
        const seed_pert = {seed_perturbation};
        const iter = {iterations};
        const points = module.lift_public_key(x_coord, seed_pert, iter);
        const score = module.score_resonance(points);
        const result = {{
            points: points.slice(0, 100),  // Send only first 100 to save bandwidth
            resonance_score: score
        }};
        console.log(JSON.stringify(result));
    }});
    """
    
    try:
        output = subprocess.check_output(['node', '-e', node_script], timeout=60)
        result = json.loads(output.decode('utf-8'))
        return result['points'], result['resonance_score']
    except subprocess.TimeoutExpired:
        print(f"[{NODE_ID}] WASM execution timed out!")
        return [], 0.0
    except Exception as e:
        print(f"[{NODE_ID}] WASM execution error: {e}")
        return [], 0.0

# ================== MQTT CALLBACKS ==================
def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print(f"[{NODE_ID}] Connected to MQTT broker at {MQTT_BROKER}:{MQTT_PORT}")
        client.subscribe(MQTT_TOPIC_TASK)
        # Announce online status
        status_payload = json.dumps({{"status": "online", "node": NODE_ID, "timestamp": time.time()}})
        client.publish(MQTT_TOPIC_STATUS, status_payload)
        print(f"[{NODE_ID}] Subscribed to {MQTT_TOPIC_TASK}. Waiting for tasks...")
    else:
        print(f"[{NODE_ID}] Connection failed with code {rc}")

def on_message(client, userdata, msg):
    try:
        task = json.loads(msg.payload)
        task_id = task['task_id']
        x_coordinate = task['x_coordinate']
        seed_perturbation = task['seed_perturbation']
        iterations = task.get('iterations', 5000)
        
        print(f"[{NODE_ID}] Received task {task_id}: x={str(x_coordinate)[:16]}... seed_pert={seed_perturbation}")
        
        # Execute the lifting
        start_time = time.time()
        points, score = run_wasm_lift(str(x_coordinate), seed_perturbation, iterations)
        elapsed = time.time() - start_time
        
        # Publish result
        result_payload = {
            'task_id': task_id,
            'node_id': NODE_ID,
            'lifted_divisor': points,
            'resonance_score': score,
            'elapsed_seconds': elapsed,
            'timestamp': time.time()
        }
        client.publish(MQTT_TOPIC_RESULT, json.dumps(result_payload))
        print(f"[{NODE_ID}] Task {task_id} completed. Score: {score:.4f} in {elapsed:.2f}s")
        
    except Exception as e:
        print(f"[{NODE_ID}] Error processing message: {e}")

# ================== MAIN LOOP ==================
client = mqtt.Client(client_id=NODE_ID)
client.username_pw_set(MQTT_USERNAME, MQTT_PASSWORD)
client.tls_set()  # Enable TLS for secure connection
client.on_connect = on_connect
client.on_message = on_message

print(f"[{NODE_ID}] Starting Aeonic Conduit Node...")
client.connect(MQTT_BROKER, MQTT_PORT, 60)
client.loop_forever()

In [ ]:
# Cell 4: Keep-Alive Heartbeat (Prevents Colab Idle Reclaim)
# This cell runs an infinite loop that sends a heartbeat every 60 seconds.
# It should be run AFTER Cell 3 starts successfully.
# IMPORTANT: Run this cell only if Cell 3 is running in the background.
# For Colab, you can use a different approach: keep the notebook alive by using JavaScript.

import time

print("Heartbeat monitor started. Sending status every 60 seconds.")
while True:
    try:
        status_payload = json.dumps({
            "status": "alive",
            "node": NODE_ID,
            "timestamp": time.time()
        })
        client.publish(MQTT_TOPIC_STATUS, status_payload)
        print(f"[{NODE_ID}] Heartbeat sent at {time.strftime('%H:%M:%S')}")
    except Exception as e:
        print(f"Heartbeat error: {e}")
    time.sleep(60)